In [1]:
import subprocess, sys
print("Installing dspy...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dspy"])
print("✓ Installed!")

Installing dspy...
✓ Installed!


In [2]:
import dspy
import os
from dotenv import load_dotenv

load_dotenv()  # reads your .env
api_key = os.getenv("LITELLM_API_KEY")

lm = dspy.LM(
    "openai/hosted_vllm/Qwen/Qwen3.6-35B-A3B-FP8",
    api_key=api_key,
    api_base="https://litellm.professor-x.de/v1",
    temperature=0.2,
)
dspy.configure(lm=lm)
print("✓ DSPy configured")

✓ DSPy configured


In [3]:
# Simplest  test: sending prompt directly through the LM
result = lm("What are the three citation intent classes in SciCite?")
print(result)

[{'text': '\n\nThe **SciCite** dataset defines three citation intent classes used to classify how scientific papers reference prior work:\n\n1. **Background**: Citations that provide context, motivation, or general knowledge related to the research problem or field.\n2. **Method**: Citations that describe methods, techniques, tools, datasets, or experimental procedures used in the work.\n3. **Result**: Citations that compare, contrast, or support the findings, outcomes, or results of the study.\n\nThese categories were introduced to enable fine-grained analysis of citation behavior in scientific literature and are widely used in NLP tasks for scientific text understanding.', 'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Understand User Query**: The user is asking about the "three citation intent classes in SciCite".\n2.  **Identify Key Entity**: SciCite is a dataset/tool for citation intent classification in scientific literature.\n3.  **Recall/Verify Knowledge**: \n   - I

In [4]:
from typing import Literal

# A DSPy Signature declares the task structure — NOT the prompt wording.
class ClassifyCitation(dspy.Signature):
    """Classify the intent of a citation in a scientific paper."""

    citation_text: str = dspy.InputField(desc="The sentence containing the citation")
    label: Literal["background", "method", "result"] = dspy.OutputField(
        desc="The citation's intent: background, method, or result"
    )

# Predict turns the signature into something runnable
classify = dspy.Predict(ClassifyCitation)
print("✓ Signature and classifier created")

✓ Signature and classifier created


In [5]:
# Test on a single citation
test_text = "We use the BERT model introduced by Devlin et al. (2019)."
result = classify(citation_text=test_text)
print("Citation:", test_text)
print("Predicted label:", result.label)

Citation: We use the BERT model introduced by Devlin et al. (2019).
Predicted label: method


In [6]:
import json
from sklearn.metrics import accuracy_score, f1_score

# Load SciCite dev data (same file your June 1 notebook used)
dev_data = []
with open("../scicite/dev.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        dev_data.append({"text": ex["string"], "label": ex["label"]})

# Take a small sample for a quick baseline (scale up later)
sample = dev_data[:50]
print(f"Testing on {len(sample)} citations...\n")

true_labels = []
pred_labels = []

for i, ex in enumerate(sample):
    pred = classify(citation_text=ex["text"]).label   # DSPy classifier
    true_labels.append(ex["label"])
    pred_labels.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  processed {i+1}/{len(sample)}")

# Score it
acc = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average="macro")

print(f"\n--- DSPy baseline (un-optimized) ---")
print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")

Testing on 50 citations...

  processed 10/50
  processed 20/50
  processed 30/50
  processed 40/50
  processed 50/50

--- DSPy baseline (un-optimized) ---
Accuracy: 0.7600
Macro F1: 0.7295


In [8]:
# The metric tells the optimizer whether a prediction is "good."
def citation_metric(example, prediction, trace=None):
    return example.label == prediction.label

print("✓ Metric defined")

✓ Metric defined


In [9]:
import dspy

# Wrap training examples as dspy.Example objects.
# Use a DIFFERENT slice than the baseline (which used [:50]).
trainset = []
for ex in dev_data[50:150]:
    trainset.append(
        dspy.Example(citation_text=ex["text"], label=ex["label"]).with_inputs("citation_text")
    )

print(f"Training set: {len(trainset)} examples")

Training set: 100 examples


In [10]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=citation_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
)

print("Optimizing... (this makes several model calls, give it a minute or two)")

optimized_classify = optimizer.compile(classify, trainset=trainset)

print("✓ Optimization complete!")

Optimizing... (this makes several model calls, give it a minute or two)


  7%|▋         | 7/100 [00:39<08:50,  5.70s/it]

Bootstrapped 4 full traces after 7 examples for up to 1 rounds, amounting to 7 attempts.
✓ Optimization complete!


In [11]:
# Re-measure the OPTIMIZED classifier on the SAME 50 examples as the baseline
print("Testing optimized classifier on same 50 citations...\n")

true_labels = []
pred_labels = []

for i, ex in enumerate(dev_data[:50]):        # same slice as baseline
    pred = optimized_classify(citation_text=ex["text"]).label   # optimized version
    true_labels.append(ex["label"])
    pred_labels.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  processed {i+1}/50")

acc_opt = accuracy_score(true_labels, pred_labels)
f1_opt = f1_score(true_labels, pred_labels, average="macro")

print(f"\n--- Comparison ---")
print(f"Baseline   : Accuracy 0.7600 | Macro F1 0.7295")
print(f"Optimized  : Accuracy {acc_opt:.4f} | Macro F1 {f1_opt:.4f}")

Testing optimized classifier on same 50 citations...

  processed 10/50
  processed 20/50
  processed 30/50
  processed 40/50
  processed 50/50

--- Comparison ---
Baseline   : Accuracy 0.7600 | Macro F1 0.7295
Optimized  : Accuracy 0.6800 | Macro F1 0.6890


In [12]:
# Larger, more reliable evaluation: both classifiers on the SAME 200 examples
eval_set = dev_data[:200]
print(f"Evaluating BOTH classifiers on {len(eval_set)} citations...")
print("This makes ~400 model calls — give it 10-20 min.\n")

base_true, base_pred = [], []
opt_true,  opt_pred  = [], []

for i, ex in enumerate(eval_set):
    # baseline classifier
    base_pred.append(classify(citation_text=ex["text"]).label)
    base_true.append(ex["label"])
    # optimized classifier
    opt_pred.append(optimized_classify(citation_text=ex["text"]).label)
    opt_true.append(ex["label"])
    if (i + 1) % 25 == 0:
        print(f"  processed {i+1}/{len(eval_set)}")

base_acc = accuracy_score(base_true, base_pred)
base_f1  = f1_score(base_true, base_pred, average="macro")
opt_acc  = accuracy_score(opt_true,  opt_pred)
opt_f1   = f1_score(opt_true,  opt_pred,  average="macro")

print(f"\n--- Comparison on {len(eval_set)} examples ---")
print(f"Baseline   : Accuracy {base_acc:.4f} | Macro F1 {base_f1:.4f}")
print(f"Optimized  : Accuracy {opt_acc:.4f} | Macro F1 {opt_f1:.4f}")

Evaluating BOTH classifiers on 200 citations...
This makes ~400 model calls — give it 10-20 min.

  processed 25/200
  processed 50/200
  processed 75/200
  processed 100/200
  processed 125/200
  processed 150/200
  processed 175/200
  processed 200/200

--- Comparison on 200 examples ---
Baseline   : Accuracy 0.7200 | Macro F1 0.6498
Optimized  : Accuracy 0.6650 | Macro F1 0.6472


In [14]:
import subprocess, sys
print("Installing optuna...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
print("✓ Installed!")

Installing optuna...
✓ Installed!


In [15]:
from dspy.teleprompt import MIPROv2

# MIPROv2 optimizes BOTH the instruction wording AND few-shot examples,
# searching with Bayesian optimization. This tests whether instruction
# rewriting (which BootstrapFewShot does NOT do) helps where BFS didn't.
mipro = MIPROv2(
    metric=citation_metric,
    auto="light",   # light/medium/heavy — "light" keeps it shorter
)

print("Optimizing with MIPROv2... this is SLOWER (many model calls). Be patient.\n")

mipro_classify = mipro.compile(
    classify,
    trainset=trainset,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    requires_permission_to_run=False,   # don't pause to ask permission
)

print("✓ MIPROv2 optimization complete!")

2026/06/16 02:39:20 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/06/16 02:39:20 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 80

2026/06/16 02:39:20 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/06/16 02:39:20 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/06/16 02:39:20 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Optimizing with MIPROv2... this is SLOWER (many model calls). Be patient.

Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 25%|██▌       | 5/20 [00:00<00:00, 23.75it/s]


Bootstrapped 3 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/6


 15%|█▌        | 3/20 [00:00<00:01, 15.54it/s]


...


Average Metric: 21.00 / 35 (60.0%): 100%|██████████| 35/35 [00:15<00:00,  2.33it/s]

2026/06/16 02:48:25 INFO dspy.evaluate.evaluate: Average Metric: 21 / 35 (60.0%)
2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 60.0 on minibatch of size 35 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 5'].
2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [60.0, 60.0, 71.43, 71.43, 68.57, 71.43, 80.0, 68.57, 77.14, 60.0]
2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [67.5, 68.75]
2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 68.75
2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 13 / 13 - Full Evaluation =====
2026/06/16 02:48:25 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 72.38) from minibatch trials...



Average Metric: 58.00 / 80 (72.5%): 100%|██████████| 80/80 [00:22<00:00,  3.63it/s]

2026/06/16 02:48:47 INFO dspy.evaluate.evaluate: Average Metric: 58 / 80 (72.5%)
2026/06/16 02:48:47 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 72.5
2026/06/16 02:48:47 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [67.5, 68.75, 72.5]
2026/06/16 02:48:47 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 72.5
2026/06/16 02:48:47 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/06/16 02:48:47 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/06/16 02:48:47 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 72.5!



✓ MIPROv2 optimization complete!


In [16]:
# Measure MIPROv2 result on the SAME 200 examples for a fair 3-way comparison
print("Testing MIPROv2-optimized classifier on same 200 citations...\n")
m_true, m_pred = [], []
for i, ex in enumerate(dev_data[:200]):
    m_pred.append(mipro_classify(citation_text=ex["text"]).label)
    m_true.append(ex["label"])
    if (i + 1) % 25 == 0:
        print(f"  processed {i+1}/200")

m_acc = accuracy_score(m_true, m_pred)
m_f1  = f1_score(m_true, m_pred, average="macro")

print(f"\n=== THREE-WAY COMPARISON (200 examples) ===")
print(f"Baseline           : Acc 0.7200 | Macro F1 0.6498")
print(f"BootstrapFewShot   : Acc 0.6650 | Macro F1 0.6472")
print(f"MIPROv2            : Acc {m_acc:.4f} | Macro F1 {m_f1:.4f}")

Testing MIPROv2-optimized classifier on same 200 citations...

  processed 25/200
  processed 50/200
  processed 75/200
  processed 100/200
  processed 125/200
  processed 150/200
  processed 175/200
  processed 200/200

=== THREE-WAY COMPARISON (200 examples) ===
Baseline           : Acc 0.7200 | Macro F1 0.6498
BootstrapFewShot   : Acc 0.6650 | Macro F1 0.6472
MIPROv2            : Acc 0.7350 | Macro F1 0.6781


In [17]:
# Save the optimized program so you never have to re-run the optimization
mipro_classify.save("../mipro_optimized_scicite.json")
print("✓ Saved optimized program")

✓ Saved optimized program


In [18]:
# Save the comparison numbers
import json
results = {
    "baseline":          {"accuracy": 0.7200, "macro_f1": 0.6498},
    "bootstrap_fewshot": {"accuracy": 0.6650, "macro_f1": 0.6472},
    "miprov2":           {"accuracy": 0.7350, "macro_f1": 0.6781},
    "eval_examples": 200,
    "taxonomy": "SciCite",
}
with open("../dspy_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("✓ Saved results")

✓ Saved results
